# Week 4 (starter): Multi-Tool Assistant

## Part 1

In [1]:
import ast, json, math, threading


def get_api_key(identifier: str = "ANTHROPIC_API_KEY") -> str:
    """Get the Anthropic API key from Colab's userdata or a .env file."""
    try:
        from google.colab import userdata
        API_KEY = userdata.get(identifier)
    except Exception:
        from dotenv import load_dotenv
        import os
        load_dotenv()
        API_KEY = os.getenv(identifier)

    return API_KEY


ANTHROPIC_API_KEY = get_api_key()
print("ANTHROPIC_API_KEY set:", bool(ANTHROPIC_API_KEY))
# Everything through the dispatch demo runs without a key. The model loop (Part 1) needs one.

ANTHROPIC_API_KEY set: True


### Tool Schemas

In [2]:
# Tool schemas

BODY_NAMES = [
    "Mercury", "Venus", "Earth", "Mars",
    "Jupiter", "Saturn", "Uranus", "Neptune",
    "Moon",
]

TOOLS = [
    {
        "name": "get_body",
        "description": (
            "Look up published physical constants for a solar system body. Covers the eight "
            "planets and Earth's Moon only; dwarf planets, other moons, asteroids and the Sun "
            "are not in this dataset. Returns mass, mean radius, surface gravity and mean "
            "orbital distance from the Sun."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "enum": BODY_NAMES,
                    "description": "Body to look up. Must be one of the listed names, capitalized exactly as shown.",
                },
            },
            "required": ["name"],
            "additionalProperties": False,
        },
    },
    {
        "name": "escape_velocity",
        "description": (
            "Compute escape velocity at the surface of a body from its mass and radius, using "
            "v = sqrt(2GM/r). This tool performs no lookup: obtain mass_kg and radius_km from "
            "get_body first and pass those values through. Returns speed in metres per second."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "mass_kg": {
                    "type": "number",
                    "minimum": 1e20,
                    "maximum": 1e29,
                    "description": "Body mass in KILOGRAMS, as returned by get_body. Do not estimate or recall this value.",
                },
                "radius_km": {
                    "type": "number",
                    "minimum": 100.0,
                    "maximum": 1e5,
                    "description": (
                        "Mean radius in KILOMETRES, as returned by get_body's radius_km field. "
                        "Pass the value unchanged; do not convert it to metres."
                    ),
                },
            },
            "required": ["mass_kg", "radius_km"],
            "additionalProperties": False,
        },
    },
    {
        "name": "run_python",
        "description": (
            "Evaluate a single arithmetic expression and return the result."
            # "Variables, function calls, imports, and attribute access are rejected so substitute concrete numbers in to the expression before calling."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "maxLength": 200,
                    "description": "Arithmetic over numeric literals, e.g. '5027.1 * 2.23694'. No names or calls.",
                },
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
]

print("tools:", [t["name"] for t in TOOLS])

tools: ['get_body', 'escape_velocity', 'run_python']


### Tools 1 and 2: get_body and escape_velocity

In [3]:
# Tools 1 and 2. Values are rounded from public sources.

G = 6.67430e-11  # gravitational constant, m^3

BODIES = {
    "Mercury": {"mass_kg": 3.301e23, "radius_km": 2439.7, "surface_gravity_m_s2": 3.70, "mean_distance_au": 0.387},
    "Venus":   {"mass_kg": 4.867e24, "radius_km": 6051.8, "surface_gravity_m_s2": 8.87, "mean_distance_au": 0.723},
    "Earth":   {"mass_kg": 5.972e24, "radius_km": 6371.0, "surface_gravity_m_s2": 9.807, "mean_distance_au": 1.000},
    "Mars":    {"mass_kg": 6.417e23, "radius_km": 3389.5, "surface_gravity_m_s2": 3.71, "mean_distance_au": 1.524},
    "Jupiter": {"mass_kg": 1.898e27, "radius_km": 69911.0, "surface_gravity_m_s2": 24.79, "mean_distance_au": 5.203},
    "Saturn":  {"mass_kg": 5.683e26, "radius_km": 58232.0, "surface_gravity_m_s2": 10.44, "mean_distance_au": 9.537},
    "Uranus":  {"mass_kg": 8.681e25, "radius_km": 25362.0, "surface_gravity_m_s2": 8.87, "mean_distance_au": 19.19},
    "Neptune": {"mass_kg": 1.024e26, "radius_km": 24622.0, "surface_gravity_m_s2": 11.15, "mean_distance_au": 30.07},
    "Moon":    {"mass_kg": 7.342e22, "radius_km": 1737.4, "surface_gravity_m_s2": 1.62, "mean_distance_au": 1.000},
}


def get_body(name):
    """Return published constants for one body. The enum makes the KeyError unreachable
    through a validated call, but the check stays: dispatch() is callable directly."""
    if name not in BODIES:
        raise KeyError(f"no body named {name!r} in the dataset")
    return {"name": name, **BODIES[name]}


def escape_velocity(mass_kg, radius_km):
    """v = sqrt(2GM/r). Takes values from get_body rather than looking them up, so that a
    correct call requires a prior retrieval."""
    v = math.sqrt(2 * G * mass_kg / (radius_km * 1000.0))
    return {"escape_velocity_m_s": round(v, 1)}


print(get_body("Mars"))
print(escape_velocity(**{k: v for k, v in BODIES["Earth"].items() if k in ("mass_kg", "radius_km")}))

{'name': 'Mars', 'mass_kg': 6.417e+23, 'radius_km': 3389.5, 'surface_gravity_m_s2': 3.71, 'mean_distance_au': 1.524}
{'escape_velocity_m_s': 11186.0}


## Part 2

### Guarded Code Runner for Tool 3

In [4]:
# Tool 3: the guarded code-runner 
# Two stages, deliberately separate: approve the whole tree, then evaluate it.

class GuardError(Exception):
    """Input rejected by the guard, before any evaluation."""


MAX_EXPR_CHARS = 200   # matches maxLength in the schema
MAX_POW_EXPONENT = 64  # bounds the cost of a single ** operation
TIME_LIMIT_S = 2.0     # backstop for anything the structural caps miss

# The allowlist. Every node in the parsed tree must be an instance of one of these.
_ALLOWED_NODES = (
    ast.Expression,
    ast.Constant,
    ast.BinOp, ast.UnaryOp,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow,
    ast.USub, ast.UAdd,
)

_OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
    ast.FloorDiv: lambda a, b: a // b,
    ast.Mod: lambda a, b: a % b,
    ast.Pow: lambda a, b: a ** b,
}
_UNARY = {ast.USub: lambda a: -a, ast.UAdd: lambda a: +a}


def _literal_number(node):
    """Return the value of a numeric literal, or None if the node is anything else.
    Used to require that ** exponents are literals rather than computed at runtime."""
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        inner = _literal_number(node.operand)
        if inner is None:
            return None
        return -inner if isinstance(node.op, ast.USub) else inner
    return None


def _approve(tree):
    """Stage 1: walk the entire tree and reject anything not on the allowlist.
    Nothing is evaluated here. A rejected expression never reaches stage 2 at all."""
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            raise GuardError(f"disallowed syntax: {type(node).__name__}")
        if isinstance(node, ast.Constant) and type(node.value) not in (int, float):
            raise GuardError(f"only int and float literals are allowed, got {type(node.value).__name__}")
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Pow):
            exponent = _literal_number(node.right)
            if exponent is None:
                raise GuardError("** exponent must be a numeric literal, not a computed value")
            if abs(exponent) > MAX_POW_EXPONENT:
                raise GuardError(f"** exponent magnitude exceeds {MAX_POW_EXPONENT}")


def _evaluate(node):
    """Stage 2: evaluate an already-approved tree."""
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_evaluate(node.left), _evaluate(node.right))
    if isinstance(node, ast.UnaryOp):
        return _UNARY[type(node.op)](_evaluate(node.operand))
    raise GuardError(f"unreachable: {type(node).__name__} passed approval")  # guards against drift


def _evaluate_with_time_limit(tree, seconds):
    """Run stage 2 in a worker thread and give up waiting after `seconds`.

    Honest limitation: a Python thread cannot be killed from outside, so this bounds how
    long the *caller* waits, not how long the work runs. That is why the exponent cap in
    stage 1 is the primary defence and this is the backstop."""
    box = {}

    def worker():
        try:
            box["value"] = _evaluate(tree.body)
        except BaseException as exc:  # re-raised on the calling thread below
            box["error"] = exc

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(seconds)
    if thread.is_alive():
        raise GuardError(f"evaluation exceeded the {seconds}s time limit")
    if "error" in box:
        raise box["error"]
    return box["value"]


def run_python(expression):
    if len(expression) > MAX_EXPR_CHARS:
        raise GuardError(f"expression exceeds {MAX_EXPR_CHARS} characters")
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError as exc:
        raise GuardError(f"could not parse expression: {exc.msg}")
    _approve(tree)                                       # stage 1
    result = _evaluate_with_time_limit(tree, TIME_LIMIT_S)  # stage 2
    if type(result) not in (int, float):                 # e.g. (-8) ** 0.5 returns complex
        raise GuardError(f"result is not a real number: {type(result).__name__}")
    if isinstance(result, float) and not math.isfinite(result):
        raise GuardError("result is not finite")
    return result


## Part 3

### Validation

In [5]:
# Validator and dispatcher. Extended from the starter to cover the constraints these
# schemas actually use: minimum, maximum, maxLength, and additionalProperties.

IMPL = {"get_body": get_body, "escape_velocity": escape_velocity, "run_python": run_python}


class ToolArgError(Exception):
    """Arguments rejected against the schema, before the implementation is called."""


def validate(name, args):
    spec = next((t["input_schema"] for t in TOOLS if t["name"] == name), None)
    if spec is None:
        raise ToolArgError(f"unknown tool: {name}")
    if not isinstance(args, dict):
        raise ToolArgError("arguments must be a JSON object")

    for field in spec.get("required", []):
        if field not in args:
            raise ToolArgError(f"missing required field: {field}")

    for key, value in args.items():
        prop = spec["properties"].get(key)
        if prop is None:
            # additionalProperties is False on every tool, so this is always an error.
            raise ToolArgError(f"unexpected field: {key} (expected one of {list(spec['properties'])})")

        if prop["type"] == "string":
            if not isinstance(value, str):
                raise ToolArgError(f"{key} must be a string, got {type(value).__name__}")
            if "maxLength" in prop and len(value) > prop["maxLength"]:
                raise ToolArgError(f"{key} exceeds maxLength {prop['maxLength']}")
        elif prop["type"] == "number":
            if type(value) not in (int, float):  # excludes bool, which is an int subclass
                raise ToolArgError(f"{key} must be a number, got {type(value).__name__}")
            if isinstance(value, float) and not math.isfinite(value):
                raise ToolArgError(f"{key} must be finite")
        else:
            raise ValueError(f"extend validate() to support schema type: {prop['type']}")

        if "enum" in prop and value not in prop["enum"]:
            raise ToolArgError(f"{key}={value!r} is not one of {prop['enum']}")
        if "minimum" in prop and value < prop["minimum"]:
            raise ToolArgError(f"{key}={value} is below minimum {prop['minimum']}")
        if "maximum" in prop and value > prop["maximum"]:
            # This is the unit-error catcher: a radius passed in metres lands here.
            raise ToolArgError(f"{key}={value} is above maximum {prop['maximum']}")


def dispatch(name, args):
    """Validate, execute, and return a JSON-serializable result the model can read.
    Errors are returned rather than raised: a model that receives an error message can
    correct itself, while an exception ends the conversation."""
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)  # reject results the model cannot receive as JSON
        return {"ok": True, "tool": name, "output": output}
    except ToolArgError as exc:
        return {"ok": False, "tool": name, "error_type": "invalid_arguments", "message": str(exc)}
    except GuardError as exc:
        return {"ok": False, "tool": name, "error_type": "guard_rejected", "message": str(exc)}
    except Exception as exc:
        return {"ok": False, "tool": name, "error_type": "execution_error", "message": str(exc)}


# Test manual tool dispatch
print("lookup: ", dispatch("get_body", {"name": "Mars"}))
print("chained:", dispatch("escape_velocity", {"mass_kg": 6.417e23, "radius_km": 3389.5}))
print("runner: ", dispatch("run_python", {"expression": "5027.1 * 2.23694"}))

lookup:  {'ok': True, 'tool': 'get_body', 'output': {'name': 'Mars', 'mass_kg': 6.417e+23, 'radius_km': 3389.5, 'surface_gravity_m_s2': 3.71, 'mean_distance_au': 1.524}}
chained: {'ok': True, 'tool': 'escape_velocity', 'output': {'escape_velocity_m_s': 5027.1}}
runner:  {'ok': True, 'tool': 'run_python', 'output': 11245.321074000001}


In [6]:
def show(name, args, note=""):
    result = dispatch(name, args)
    tag = "OK " if result["ok"] else "ERR"
    suffix = f"   # {note}" if note else ""
    print(f"[{tag}] {name}({json.dumps(args)})")
    print(f"      -> {json.dumps(result)}{suffix}\n")


print("--- the three-link chain, run by hand ---\n")
body = dispatch("get_body", {"name": "Mars"})["output"]
show("get_body", {"name": "Mars"}, "step 1: retrieve")
show("escape_velocity", {"mass_kg": body["mass_kg"], "radius_km": body["radius_km"]},
     "step 2: uses step 1's fields")
show("run_python", {"expression": "5027.1 * 2.23694"},
     "step 3: m/s -> mph, uses step 2's result")

print("--- error paths the model is likely to find ---\n")
show("get_body", {"name": "Pluto"}, "not in the enum")
show("escape_velocity", {"mass_kg": 5.972e24, "radius_km": 6371000},
     "radius in metres: caught by maximum, not silently wrong")
show("escape_velocity", {"mass_kg": "5.972e24", "radius_km": 6371.0},
     "number sent as a string")
show("run_python", {"expression": "6371 * 2 * math.pi"}, "a name the guard will not allow")

--- the three-link chain, run by hand ---

[OK ] get_body({"name": "Mars"})
      -> {"ok": true, "tool": "get_body", "output": {"name": "Mars", "mass_kg": 6.417e+23, "radius_km": 3389.5, "surface_gravity_m_s2": 3.71, "mean_distance_au": 1.524}}   # step 1: retrieve

[OK ] escape_velocity({"mass_kg": 6.417e+23, "radius_km": 3389.5})
      -> {"ok": true, "tool": "escape_velocity", "output": {"escape_velocity_m_s": 5027.1}}   # step 2: uses step 1's fields

[OK ] run_python({"expression": "5027.1 * 2.23694"})
      -> {"ok": true, "tool": "run_python", "output": 11245.321074000001}   # step 3: m/s -> mph, uses step 2's result

--- error paths the model is likely to find ---

[ERR] get_body({"name": "Pluto"})
      -> {"ok": false, "tool": "get_body", "error_type": "invalid_arguments", "message": "name='Pluto' is not one of ['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Moon']"}   # not in the enum

[ERR] escape_velocity({"mass_kg": 5.972e+24, "radius_km

### Evaluation

In [7]:
from anthropic import Anthropic

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)

CLAUDE_MODEL = "claude-haiku-4-5"

CALL_LOG = []  # (turn, tool, args, result) — the evidence for Parts 3 and 4
PRN_SEP = "-" * 24 + "\n"


def run_conversation(query, max_turns=6, verbose=True):
    """Send `query` with TOOLS available, execute every tool call the model makes, feed the
    results back, and continue until the model answers in prose. Returns the final text."""
    current_turn = 0  # turn cap counter

    messages = [
        {"role": "user", "content": query}
    ]

    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        tools=TOOLS,
        messages=messages
    )

    if verbose and response.stop_reason != "tool_use":
        # A turn that calls nothing is itself a finding, not a non-event.
        print(f"  [no tool call] stop_reason={response.stop_reason}")

    while response.stop_reason == "tool_use" and current_turn < max_turns:
        # add assistant turn
        messages.append({"role": "assistant", "content": response.content})

        # execute tool calls and accumulate results
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = dispatch(block.name, block.input)

                tool_result = {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
                # flag result as error
                if not result["ok"]:
                    tool_result["is_error"] = True

                tool_results.append(tool_result)
                # append to the call log 
                CALL_LOG.append((current_turn, block.name, block.input, result))

                if verbose:
                    tag = "OK " if result["ok"] else "ERR"
                    print(f"  [{tag}] turn {current_turn}: {block.name}({json.dumps(block.input)})")
                    if not result["ok"]:
                        print(f"        -> {result['error_type']}: {result['message']}")

        # send results back as a user turn
        messages.append({"role": "user", "content": tool_results})
        # continue the conversation
        response = anthropic_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=4096,
            tools=TOOLS,
            messages=messages
        )
        current_turn += 1

    # after stop reason isn't tool_use return the answer
    for block in response.content:
        if block.type == "text":
            return block.text

    # turn cap or a max_tokens truncation
    return f"<no text returned: stop_reason={response.stop_reason}, turns={current_turn}>"


def ask(query, max_turns=6):
    """Run one query, print its calls and answer, and return this query's call log."""
    print(f"> {query}")
    CALL_LOG.clear()
    answer = run_conversation(query, max_turns=max_turns)
    calls = list(CALL_LOG)
    print(f"\n{answer}\n")
    print(PRN_SEP)
    return calls

In [8]:
# Part 3: three queries covering all three tools

prompts = [
    "How strong is gravity on Titan compared to Earth?",
    "What would a 185 pound person weigh on Mars?",
    "What is the escape velocity of Jupiter in mph?",
    "What is Mars's escape velocity in km/s, rounded to two decimal places?",
]

prompt_logs = {}
for prompt in prompts:
    prompt_logs[prompt] = ask(prompt)


> How strong is gravity on Titan compared to Earth?
  [no tool call] stop_reason=end_turn

I don't have data for Titan in my available tools - I can only look up information for the eight planets and Earth's Moon. Titan is one of Saturn's moons, which isn't in my dataset.

However, I can tell you about Earth's gravity. Would you like me to calculate Earth's surface gravity for comparison? If you can provide Titan's mass and radius (or its surface gravity), I could help you compare them.

For reference, Titan's surface gravity is approximately 1.35 m/s² (about 14% of Earth's), but this is general knowledge rather than from my available tools.

------------------------

> What would a 185 pound person weigh on Mars?
  [OK ] turn 0: get_body({"name": "Mars"})
  [OK ] turn 1: run_python({"expression": "185 * (3.71 / 9.81)"})

A 185-pound person would weigh approximately **70 pounds on Mars**.

This is because Mars has about 38% of Earth's surface gravity (3.71/9.81 ≈ 0.378), so a person wo

 ## Part 4

I noticed two interesting cases during my trials that aren't reliably reproducible.

When responding to the first prompt the assistant will sometimes notify the user that it's tool does not include information about Titan while other times it will rely on it's own knowledge without notifying the user. This may not be a failure case in the strictest sense, but I thought it was a noteworthy consideration.

> How strong is gravity on Titan compared to Earth?
```
  [OK ] turn 0: get_body({"name": "Earth"})

**Earth's surface gravity is 9.807 m/s²** (what we typically call 1 g).

**Titan's surface gravity is approximately 1.35 m/s²**, which is about **14% of Earth's gravity** (or 0.14 g)...
```
 
 The more interesting case is that occassionaly when answering the fourth prompt the model attempts to pass a function call to run_python which is rejected by the code runners guards:

> *"What is Mars's escape velocity in km/s, rounded to two decimal places?"*

 ```
[OK ] turn 0: get_body({"name": "Mars"})
[OK ] turn 1: escape_velocity({"mass_kg": 6.417e+23, "radius_km": 3389.5})
[ERR] turn 2: run_python({"expression": "round(5027.1 / 1000, 2)"})
      -> guard_rejected: disallowed syntax: Call
[OK ] turn 3: run_python({"expression": "5027.1 / 1000"})
```

The detailed error message allows the model to self-correct effectively and the issue is easily fixed by improving the description of the run_python function in the tool schema to include the guard parameters by adding the following line:


> "Variables, function calls, imports, and attribute access are rejected so substitute concrete numbers in to the expression before calling."

